# 🎙️ ZimanPro Voice Lab — دەنگێ تە ب سێ زمانان

ئەڤ نوتبووکە ب نموونەیەکا کورت ژ دەنگێ تە (۱۰–۲۰ چرکە) هەمی پەیڤێن وانەیان ب **دەنگێ تە** دروست دکەت:
**بادینی 🟢 · ئەلمانی 🇩🇪 · ئینگلیزی 🇬🇧**

**پێش دەستپێکرنێ:** `Runtime ➔ Change runtime type ➔ T4 GPU` هەلبژێرە.

📱 **ل سەر مۆبایلێ:** ب Chrome ڤەکە و «Desktop site» ڤەکە (⋮ ➔ Desktop site). ل سەر هەر خانەیەکێ ▶ بکە، ئێک ل دویف ئێکێ، و چاڤەڕێ بە تا ✅ دیار دبیت.

> ⚠️ تنێ دەنگێ خۆ کلۆن بکە. مۆدێلێ OmniVoice ب لیسانسا CC-BY-NC یە (بۆ بکارئینانا نە-بازرگانی).

## ١. دامەزراندن

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install omnivoice soundfile huggingface_hub accelerate

## ٢. فایلێن Voice Lab

In [ ]:
%%writefile voice_core.py
"""
ZimanPro Voice Lab — ناڤەندا کلۆنکرنا دەنگی

ب نموونەیەکا کورت ژ دەنگێ تە (~١٠–٢٠ چرکە)، هەمی پەیڤێن وانەیان ب دەنگێ تە
ب سێ زمانان دروست دکەت:
  - ku  (بادینی)  ➔ Dezheen89/omnivoice-badini  (fine-tune ل سەر بادینی)
  - de / en        ➔ k2-fsa/OmniVoice            (مۆدێلێ گشتی، 600+ زمان)

پێدڤی ب GPU (NVIDIA) یە — Google Colab یا بێ پارە (T4) بەسە.
تێبینی: مۆدێلێ OmniVoice ب لیسانسا CC-BY-NC یە (بۆ بکارئینانا نە-بازرگانی).
"""
from __future__ import annotations

import hashlib
import os
import subprocess
import tempfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import soundfile as sf

SAMPLE_RATE = 24000
BASE_MODEL = os.environ.get("ZIMAN_BASE_MODEL", "k2-fsa/OmniVoice")
BADINI_MODEL = os.environ.get("ZIMAN_BADINI_MODEL", "Dezheen89/omnivoice-badini")

# دەقێن نموونە کو بەکارهێنەر دخوینیت (هەمان دەق د ئەپی دا هەنە)
REFERENCE_SCRIPTS = {
    "ku": "سلاڤ، ئەڤە دەنگێ منە. ئەز ب کوردی دئاخڤم و ئەز دێ ئەلمانی و ئینگلیزی فێر بم. ئەڤرۆ هەوا خۆشە و ئەز زۆر کەیفخۆشم.",
    "de": "Hallo, das ist meine Stimme. Ich lerne jeden Tag ein bisschen Deutsch, und das macht mir Spaß.",
    "en": "Hello, this is my voice. I am learning English a little every day, and I really enjoy it.",
}


@dataclass
class Reference:
    """نموونەیا دەنگی + دەقێ وێ"""
    audio_path: str
    text: str


def to_wav24k(src: str, dst: str) -> str:
    """هەر فایلەکێ دەنگی (m4a, mp3, ogg, wav...) دکەتە WAV 24kHz mono و بێدەنگیێ ژێ دبەت"""
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", src,
         "-af", "silenceremove=start_periods=1:start_threshold=-45dB:"
                "stop_periods=-1:stop_duration=0.6:stop_threshold=-45dB,loudnorm",
         "-ac", "1", "-ar", str(SAMPLE_RATE), dst],
        check=True,
    )
    return dst


def wav_to_mp3(samples: np.ndarray, dst: str) -> str:
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        sf.write(tmp.name, samples, SAMPLE_RATE)
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", tmp.name,
             "-af", "loudnorm", "-codec:a", "libmp3lame", "-q:a", "4", dst],
            check=True,
        )
    finally:
        os.unlink(tmp.name)
    return dst


class VoiceCloner:
    """دوو مۆدێلان ب شێوەیەکێ تەمبەل (lazy) بار دکەت"""

    def __init__(self, device: str = "cuda:0"):
        self.device = device
        self._models: dict[str, object] = {}

    def _model(self, lang: str):
        name = BADINI_MODEL if lang == "ku" else BASE_MODEL
        if name not in self._models:
            import torch
            from omnivoice import OmniVoice

            print(f"⏳ Loading {name} ...")
            self._models[name] = OmniVoice.from_pretrained(
                name, device_map=self.device, dtype=torch.float16
            )
        return self._models[name]

    def synthesize(self, text: str, lang: str, ref: Reference) -> np.ndarray:
        model = self._model(lang)
        kwargs = dict(text=text, ref_audio=ref.audio_path, ref_text=ref.text)
        if lang == "ku":
            # مۆدێلێ بادینی کۆدێ زمانێ kmr ب کار دئینیت
            try:
                out = model.generate(language="kmr", **kwargs)
            except TypeError:
                out = model.generate(**kwargs)
        else:
            out = model.generate(**kwargs)
        audio = out[0] if isinstance(out, (list, tuple)) else out
        return np.asarray(audio, dtype=np.float32)


def pick_reference(lang: str, refs: dict[str, Reference]) -> Reference:
    """ئەگەر نموونە ب هەمان زمانی هەبیت باشترە، دەنا نموونەیا بادینی (cross-lingual)"""
    return refs.get(lang) or refs["ku"]


def cache_key(speaker: str, lang: str, text: str) -> str:
    return hashlib.sha1(f"{speaker}|{lang}|{text}".encode()).hexdigest()[:16]


def clean_text(text: str) -> str:
    # نیشانێن «...» بۆ TTS ب کار نائێن
    return text.replace("...", " ").strip()


def build_pack(curriculum: dict, refs: dict[str, Reference], out_dir: Path,
               cloner: VoiceCloner, langs=("ku", "de", "en"), skip_existing=True) -> int:
    """هەمی پەیڤان دروست دکەت ➔ out_dir/{lang}_{id}.mp3"""
    out_dir.mkdir(parents=True, exist_ok=True)
    field = {"ku": "ku", "de": "de", "en": "en"}
    concepts = [c for u in curriculum["units"] for c in u["concepts"]]
    done = 0
    total = len(concepts) * len(langs)
    for lang in langs:
        ref = pick_reference(lang, refs)
        for c in concepts:
            dst = out_dir / f"{lang}_{c['id']}.mp3"
            done += 1
            if skip_existing and dst.exists():
                continue
            text = clean_text(c[field[lang]])
            audio = cloner.synthesize(text, lang, ref)
            wav_to_mp3(audio, str(dst))
            print(f"[{done}/{total}] {dst.name}  ←  {text}")
    return done

In [ ]:
%%writefile generate_pack.py
"""
دروستکرنا پاکێتا دەنگی ب دەنگێ تە بۆ هەمی وانەیان.

    python generate_pack.py --ref-ku my_voice_ku.m4a
    python generate_pack.py --ref-ku ku.m4a --ref-de de.m4a --ref-en en.m4a

ئەنجام: voice_pack.zip ➔ ڤەکە د ناڤ zimanpro/assets/voices/ دا و ئەپی ژ نوو build بکە.
ئەگەر تە دەقەکێ دی خواند (نە دەقێ نموونە)، ب --ref-text-ku بنڤیسە.
"""
import argparse
import json
import shutil
import tempfile
from pathlib import Path

from voice_core import REFERENCE_SCRIPTS, Reference, VoiceCloner, build_pack, to_wav24k

HERE = Path(__file__).parent


def main():
    ap = argparse.ArgumentParser()
    for lang in ("ku", "de", "en"):
        ap.add_argument(f"--ref-{lang}", help=f"نموونەیا دەنگی ب زمانێ {lang}", required=(lang == "ku"))
        ap.add_argument(f"--ref-text-{lang}", default=REFERENCE_SCRIPTS[lang])
    ap.add_argument("--curriculum", default=str(HERE / "curriculum.json"))
    ap.add_argument("--out", default="voice_pack")
    ap.add_argument("--langs", default="ku,de,en")
    args = ap.parse_args()

    work = Path(tempfile.mkdtemp())
    refs = {}
    for lang in ("ku", "de", "en"):
        src = getattr(args, f"ref_{lang}")
        if src:
            wav = to_wav24k(src, str(work / f"ref_{lang}.wav"))
            refs[lang] = Reference(wav, getattr(args, f"ref_text_{lang}"))
            print(f"✅ Reference {lang}: {src}")

    curriculum = json.loads(Path(args.curriculum).read_text(encoding="utf-8"))
    out_dir = Path(args.out)
    build_pack(curriculum, refs, out_dir, VoiceCloner(), langs=tuple(args.langs.split(",")))
    zip_path = shutil.make_archive(str(out_dir), "zip", out_dir)
    print(f"\n🎉 {zip_path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile server.py
"""
سێرڤەرێ کلۆنکرنا دەنگی بۆ ئەپێ ZimanPro.

    pip install fastapi uvicorn python-multipart
    ZIMAN_TOKEN=mysecret uvicorn server:app --host 0.0.0.0 --port 8000

ئەپ:
  1. POST /speakers   ➔ نموونەیا دەنگی فرێ دکەت، speaker_id وەردگریت
  2. POST /tts        ➔ بۆ هەر پەیڤەکێ mp3 ب دەنگێ تە وەردگریت (cache دبیت)

پاراستن: ئەگەر ZIMAN_TOKEN هاتبیتە دانان، هەر داخوازەک دڤێت سەرێ X-Token هەبیت.
"""
import json
import os
import uuid
from pathlib import Path

from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import FileResponse
from pydantic import BaseModel

from voice_core import (REFERENCE_SCRIPTS, Reference, VoiceCloner, cache_key, clean_text,
                        pick_reference, to_wav24k, wav_to_mp3)

DATA = Path(os.environ.get("ZIMAN_DATA", "ziman_data"))
SPEAKERS = DATA / "speakers"
CACHE = DATA / "cache"
SPEAKERS.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
TOKEN = os.environ.get("ZIMAN_TOKEN")
MAX_UPLOAD = 15 * 1024 * 1024  # 15MB

app = FastAPI(title="ZimanPro Voice Lab")
cloner = VoiceCloner()


def auth(x_token: str | None = Header(default=None)):
    if TOKEN and x_token != TOKEN:
        raise HTTPException(401, "bad token")


def load_refs(speaker_id: str) -> dict[str, Reference]:
    if not speaker_id.replace("-", "").isalnum():
        raise HTTPException(400, "bad speaker id")
    meta = SPEAKERS / speaker_id / "meta.json"
    if not meta.exists():
        raise HTTPException(404, "speaker not found")
    m = json.loads(meta.read_text(encoding="utf-8"))
    return {lang: Reference(str(SPEAKERS / speaker_id / f"ref_{lang}.wav"), text)
            for lang, text in m["texts"].items()}


@app.get("/health")
def health():
    return {"ok": True}


async def _save(upload: UploadFile, dst: Path):
    data = await upload.read()
    if len(data) > MAX_UPLOAD:
        raise HTTPException(413, "file too large")
    dst.write_bytes(data)


@app.post("/speakers", dependencies=[Depends(auth)])
async def create_speaker(
    consent: bool = Form(...),
    ref_ku: UploadFile = File(...),
    ref_text_ku: str = Form(REFERENCE_SCRIPTS["ku"]),
    ref_de: UploadFile | None = File(None),
    ref_text_de: str = Form(REFERENCE_SCRIPTS["de"]),
    ref_en: UploadFile | None = File(None),
    ref_text_en: str = Form(REFERENCE_SCRIPTS["en"]),
):
    # تنێ دەنگێ خۆ — نە دەنگێ کەسەکێ دی بێ دەستووری
    if not consent:
        raise HTTPException(400, "consent required: only clone your own voice")
    sid = str(uuid.uuid4())
    folder = SPEAKERS / sid
    folder.mkdir(parents=True)
    texts = {}
    for lang, up, text in (("ku", ref_ku, ref_text_ku), ("de", ref_de, ref_text_de), ("en", ref_en, ref_text_en)):
        if up is None:
            continue
        raw = folder / f"raw_{lang}"
        await _save(up, raw)
        to_wav24k(str(raw), str(folder / f"ref_{lang}.wav"))
        raw.unlink()
        texts[lang] = text
    (folder / "meta.json").write_text(json.dumps({"texts": texts}, ensure_ascii=False), encoding="utf-8")
    return {"speaker_id": sid, "langs": list(texts)}


class TtsRequest(BaseModel):
    speaker_id: str
    lang: str  # ku | de | en
    text: str


@app.post("/tts", dependencies=[Depends(auth)])
def tts(req: TtsRequest):
    if req.lang not in ("ku", "de", "en"):
        raise HTTPException(400, "lang must be ku, de or en")
    text = clean_text(req.text)
    if not text or len(text) > 300:
        raise HTTPException(400, "text empty or too long")
    refs = load_refs(req.speaker_id)
    dst = CACHE / f"{cache_key(req.speaker_id, req.lang, text)}.mp3"
    if not dst.exists():
        audio = cloner.synthesize(text, req.lang, pick_reference(req.lang, refs))
        wav_to_mp3(audio, str(dst))
    return FileResponse(dst, media_type="audio/mpeg")


@app.delete("/speakers/{speaker_id}", dependencies=[Depends(auth)])
def delete_speaker(speaker_id: str):
    """ژێبرنا دەنگێ تە ژ سێرڤەری"""
    load_refs(speaker_id)
    import shutil
    shutil.rmtree(SPEAKERS / speaker_id)
    return {"deleted": speaker_id}

In [ ]:
%%writefile curriculum.json
{
 "version": 2,
 "units": [
  {
   "id": "unit_1",
   "titleKu": "قۆناغا ١: سلاڤ و بنەڕەت",
   "titleDe": "Einheit 1: Begrüßung",
   "titleEn": "Unit 1: Greetings",
   "icon": "waving_hand",
   "color": "#58CC02",
   "concepts": [
    {
     "id": "g1",
     "ku": "سلاڤ",
     "kuLat": "Silav",
     "de": "Hallo",
     "deHint": "هالۆ",
     "en": "Hello",
     "enHint": "هێلۆ"
    },
    {
     "id": "g2",
     "ku": "سپێدە باش",
     "kuLat": "Spêde baş",
     "de": "Guten Morgen",
     "deHint": "گوتن مۆرگن",
     "en": "Good morning",
     "enHint": "گود مۆرنینگ"
    },
    {
     "id": "g3",
     "ku": "تو چاوانی؟",
     "kuLat": "Tu çawanî?",
     "de": "Wie geht es dir?",
     "deHint": "ڤی گێت ئێس دیر؟ (dir = Dativ)",
     "en": "How are you?",
     "enHint": "هاو ئار یوو؟"
    },
    {
     "id": "g4",
     "ku": "ئەز باشم",
     "kuLat": "Ez başim",
     "de": "Mir geht es gut",
     "deHint": "میر گێت ئێس گوت",
     "en": "I am fine",
     "enHint": "ئای ئێم فاین"
    },
    {
     "id": "g5",
     "ku": "زۆر سوپاس",
     "kuLat": "Zor supas",
     "de": "Vielen Dank",
     "deHint": "فیلن دانک",
     "en": "Thank you very much",
     "enHint": "سانک یوو ڤێری مەچ"
    },
    {
     "id": "g6",
     "ku": "ب خاترا تە",
     "kuLat": "Bi xatra te",
     "de": "Tschüss",
     "deHint": "چووس",
     "en": "Goodbye",
     "enHint": "گودبای"
    },
    {
     "id": "g7",
     "ku": "بەلێ",
     "kuLat": "Belê",
     "de": "Ja",
     "deHint": "یا",
     "en": "Yes",
     "enHint": "یێس"
    },
    {
     "id": "g8",
     "ku": "نەخێر",
     "kuLat": "Nexêr",
     "de": "Nein",
     "deHint": "ناین",
     "en": "No",
     "enHint": "نۆ"
    }
   ]
  },
  {
   "id": "unit_2",
   "titleKu": "قۆناغا ٢: ژمارە",
   "titleDe": "Einheit 2: Zahlen",
   "titleEn": "Unit 2: Numbers",
   "icon": "pin",
   "color": "#1CB0F6",
   "concepts": [
    {
     "id": "n1",
     "ku": "یەک",
     "kuLat": "Yek",
     "de": "eins",
     "deHint": "ئاینس",
     "en": "one",
     "enHint": "وەن"
    },
    {
     "id": "n2",
     "ku": "دوو",
     "kuLat": "Du",
     "de": "zwei",
     "deHint": "تسڤای",
     "en": "two",
     "enHint": "توو"
    },
    {
     "id": "n3",
     "ku": "سێ",
     "kuLat": "Sê",
     "de": "drei",
     "deHint": "درای",
     "en": "three",
     "enHint": "سری"
    },
    {
     "id": "n4",
     "ku": "چار",
     "kuLat": "Çar",
     "de": "vier",
     "deHint": "فیر",
     "en": "four",
     "enHint": "فۆر"
    },
    {
     "id": "n5",
     "ku": "پێنج",
     "kuLat": "Pênc",
     "de": "fünf",
     "deHint": "فوونف",
     "en": "five",
     "enHint": "فایڤ"
    },
    {
     "id": "n6",
     "ku": "دەه",
     "kuLat": "Deh",
     "de": "zehn",
     "deHint": "تسێن",
     "en": "ten",
     "enHint": "تێن"
    }
   ]
  },
  {
   "id": "unit_3",
   "titleKu": "قۆناغا ٣: مالبات",
   "titleDe": "Einheit 3: Familie",
   "titleEn": "Unit 3: Family",
   "icon": "family",
   "color": "#FF9600",
   "concepts": [
    {
     "id": "f1",
     "ku": "دایک",
     "kuLat": "Dayk",
     "de": "die Mutter",
     "deHint": "دی موتەر (die = مێ)",
     "en": "mother",
     "enHint": "مەذەر"
    },
    {
     "id": "f2",
     "ku": "باب",
     "kuLat": "Bab",
     "de": "der Vater",
     "deHint": "دێر فاتەر (der = نێر)",
     "en": "father",
     "enHint": "فاذەر"
    },
    {
     "id": "f3",
     "ku": "برا",
     "kuLat": "Bira",
     "de": "der Bruder",
     "deHint": "دێر برودەر",
     "en": "brother",
     "enHint": "برەذەر"
    },
    {
     "id": "f4",
     "ku": "خویشک",
     "kuLat": "Xweşk",
     "de": "die Schwester",
     "deHint": "دی شڤێستەر",
     "en": "sister",
     "enHint": "سیستەر"
    },
    {
     "id": "f5",
     "ku": "زارۆک",
     "kuLat": "Zarok",
     "de": "das Kind",
     "deHint": "داس کیند (das = بێلایەن)",
     "en": "child",
     "enHint": "چایلد"
    },
    {
     "id": "f6",
     "ku": "مالبات",
     "kuLat": "Malbat",
     "de": "die Familie",
     "deHint": "دی فامیلیە",
     "en": "family",
     "enHint": "فامیلی"
    }
   ]
  },
  {
   "id": "unit_4",
   "titleKu": "قۆناغا ٤: خوارن و بازاڕ",
   "titleDe": "Einheit 4: Essen & Einkaufen",
   "titleEn": "Unit 4: Food & Shopping",
   "icon": "shopping",
   "color": "#CE82FF",
   "concepts": [
    {
     "id": "s1",
     "ku": "ئاڤ",
     "kuLat": "Av",
     "de": "das Wasser",
     "deHint": "داس ڤاسەر",
     "en": "water",
     "enHint": "ووتەر"
    },
    {
     "id": "s2",
     "ku": "نان",
     "kuLat": "Nan",
     "de": "das Brot",
     "deHint": "داس برۆت",
     "en": "bread",
     "enHint": "برێد"
    },
    {
     "id": "s3",
     "ku": "چا",
     "kuLat": "Ça",
     "de": "der Tee",
     "deHint": "دێر تێ",
     "en": "tea",
     "enHint": "تی"
    },
    {
     "id": "s4",
     "ku": "ئەز حەز ژ قەهوێ دکەم",
     "kuLat": "Ez hez ji qehwê dikem",
     "de": "Ich mag Kaffee",
     "deHint": "ئیش ماگ کافێ",
     "en": "I like coffee",
     "enHint": "ئای لایک کۆفی"
    },
    {
     "id": "s5",
     "ku": "ئەز برسیمە",
     "kuLat": "Ez birsîme",
     "de": "Ich habe Hunger",
     "deHint": "ئیش هابە هونگەر",
     "en": "I am hungry",
     "enHint": "ئای ئێم هەنگری"
    },
    {
     "id": "s6",
     "ku": "ئەڤە ب چەندێ یە؟",
     "kuLat": "Eve bi çendê ye?",
     "de": "Was kostet das?",
     "deHint": "ڤاس کۆستەت داس؟",
     "en": "How much is this?",
     "enHint": "هاو مەچ ئیز ذیس؟"
    }
   ]
  },
  {
   "id": "unit_5",
   "titleKu": "قۆناغا ٥: جه و ئاراستە",
   "titleDe": "Einheit 5: Orte",
   "titleEn": "Unit 5: Places",
   "icon": "city",
   "color": "#FF4B4B",
   "concepts": [
    {
     "id": "p1",
     "ku": "مال",
     "kuLat": "Mal",
     "de": "das Haus",
     "deHint": "داس هاوس",
     "en": "house",
     "enHint": "هاوس"
    },
    {
     "id": "p2",
     "ku": "قوتابخانە",
     "kuLat": "Qutabxane",
     "de": "die Schule",
     "deHint": "دی شولە",
     "en": "school",
     "enHint": "سکوول"
    },
    {
     "id": "p3",
     "ku": "کار",
     "kuLat": "Kar",
     "de": "die Arbeit",
     "deHint": "دی ئاربایت",
     "en": "work",
     "enHint": "وۆرک"
    },
    {
     "id": "p4",
     "ku": "نەخۆشخانە",
     "kuLat": "Nexweşxane",
     "de": "das Krankenhaus",
     "deHint": "داس کرانکەنهاوس",
     "en": "hospital",
     "enHint": "هۆسپیتەل"
    },
    {
     "id": "p5",
     "ku": "تەوالێت ل کیرێ یە؟",
     "kuLat": "Tewalêt li kîrê ye?",
     "de": "Wo ist die Toilette?",
     "deHint": "ڤۆ ئیست دی تۆالێتە؟",
     "en": "Where is the toilet?",
     "enHint": "وێر ئیز ذە تۆیلێت؟"
    },
    {
     "id": "p6",
     "ku": "ئەز هاریکاریا تە دکەم",
     "kuLat": "Ez harîkariya te dikem",
     "de": "Ich helfe dir",
     "deHint": "ئیش هێلفە دیر (helfen + Dativ)",
     "en": "I help you",
     "enHint": "ئای هێلپ یوو"
    }
   ]
  },
  {
   "id": "unit_6",
   "titleKu": "قۆناغا ٦: دەم و ئاخفتن",
   "titleDe": "Einheit 6: Zeit & Gespräch",
   "titleEn": "Unit 6: Time & Talk",
   "icon": "schedule",
   "color": "#2B70C9",
   "concepts": [
    {
     "id": "t1",
     "ku": "ئەڤرۆ",
     "kuLat": "Evro",
     "de": "heute",
     "deHint": "هۆیتە",
     "en": "today",
     "enHint": "تودەی"
    },
    {
     "id": "t2",
     "ku": "سوبەهی",
     "kuLat": "Sibehî",
     "de": "morgen",
     "deHint": "مۆرگن",
     "en": "tomorrow",
     "enHint": "تومۆرۆ"
    },
    {
     "id": "t3",
     "ku": "دوهی",
     "kuLat": "Duhî",
     "de": "gestern",
     "deHint": "گێستەرن",
     "en": "yesterday",
     "enHint": "یێستەردەی"
    },
    {
     "id": "t4",
     "ku": "ئەز تێناگەهم",
     "kuLat": "Ez tênagehim",
     "de": "Ich verstehe nicht",
     "deHint": "ئیش فێرشتێهە نیشت",
     "en": "I do not understand",
     "enHint": "ئای دوو نۆت ئەندەرستاند"
    },
    {
     "id": "t5",
     "ku": "هێدی باخڤە، هیڤییە",
     "kuLat": "Hêdî baxive, hîvîye",
     "de": "Bitte langsam sprechen",
     "deHint": "بیتە لانگزام شپرێشن",
     "en": "Please speak slowly",
     "enHint": "پلیز سپیک سلۆلی"
    },
    {
     "id": "t6",
     "ku": "ناڤێ من ئازادە",
     "kuLat": "Navê min Azad e",
     "de": "Ich heiße Azad",
     "deHint": "ئیش هایسە ئازاد",
     "en": "My name is Azad",
     "enHint": "مای نەیم ئیز ئازاد"
    }
   ]
  },
  {
   "id": "unit_7",
   "titleKu": "قۆناغا ٧: رەنگ",
   "titleDe": "Einheit 7: Farben",
   "titleEn": "Unit 7: Colors",
   "icon": "palette",
   "color": "#FFC800",
   "concepts": [
    {
     "id": "k1",
     "ku": "سۆر",
     "kuLat": "Sor",
     "de": "rot",
     "deHint": "رۆت",
     "en": "red",
     "enHint": "رێد"
    },
    {
     "id": "k2",
     "ku": "شین",
     "kuLat": "Şîn",
     "de": "blau",
     "deHint": "بلاو",
     "en": "blue",
     "enHint": "بلوو"
    },
    {
     "id": "k3",
     "ku": "کەسک",
     "kuLat": "Kesk",
     "de": "grün",
     "deHint": "گروون",
     "en": "green",
     "enHint": "گرین"
    },
    {
     "id": "k4",
     "ku": "زەر",
     "kuLat": "Zer",
     "de": "gelb",
     "deHint": "گێلب",
     "en": "yellow",
     "enHint": "یێلۆ"
    },
    {
     "id": "k5",
     "ku": "ڕەش",
     "kuLat": "Reş",
     "de": "schwarz",
     "deHint": "شڤارتس",
     "en": "black",
     "enHint": "بلاک"
    },
    {
     "id": "k6",
     "ku": "سپی",
     "kuLat": "Spî",
     "de": "weiß",
     "deHint": "ڤایس",
     "en": "white",
     "enHint": "وایت"
    }
   ]
  },
  {
   "id": "unit_8",
   "titleKu": "قۆناغا ٨: لەش",
   "titleDe": "Einheit 8: Körper",
   "titleEn": "Unit 8: Body",
   "icon": "body",
   "color": "#FF86D0",
   "concepts": [
    {
     "id": "b1",
     "ku": "سەر",
     "kuLat": "Ser",
     "de": "der Kopf",
     "deHint": "دێر کۆپف",
     "en": "head",
     "enHint": "هێد"
    },
    {
     "id": "b2",
     "ku": "دەست",
     "kuLat": "Dest",
     "de": "die Hand",
     "deHint": "دی هاند",
     "en": "hand",
     "enHint": "هاند"
    },
    {
     "id": "b3",
     "ku": "چاڤ",
     "kuLat": "Çav",
     "de": "das Auge",
     "deHint": "داس ئاوگە",
     "en": "eye",
     "enHint": "ئای"
    },
    {
     "id": "b4",
     "ku": "پێ",
     "kuLat": "Pê",
     "de": "der Fuß",
     "deHint": "دێر فووس",
     "en": "foot",
     "enHint": "فووت"
    },
    {
     "id": "b5",
     "ku": "دل",
     "kuLat": "Dil",
     "de": "das Herz",
     "deHint": "داس هێرتس",
     "en": "heart",
     "enHint": "هارت"
    },
    {
     "id": "b6",
     "ku": "دەڤ",
     "kuLat": "Dev",
     "de": "der Mund",
     "deHint": "دێر موند",
     "en": "mouth",
     "enHint": "ماوس"
    }
   ]
  },
  {
   "id": "unit_9",
   "titleKu": "قۆناغا ٩: کار و کریار",
   "titleDe": "Einheit 9: Verben",
   "titleEn": "Unit 9: Verbs",
   "icon": "run",
   "color": "#00CD9C",
   "concepts": [
    {
     "id": "v1",
     "ku": "ئەز دچم",
     "kuLat": "Ez diçim",
     "de": "Ich gehe",
     "deHint": "ئیش گێهە",
     "en": "I go",
     "enHint": "ئای گۆ"
    },
    {
     "id": "v2",
     "ku": "ئەز دخۆم",
     "kuLat": "Ez dixom",
     "de": "Ich esse",
     "deHint": "ئیش ئێسە",
     "en": "I eat",
     "enHint": "ئای ئیت"
    },
    {
     "id": "v3",
     "ku": "ئەز ڤەدخۆم",
     "kuLat": "Ez vedixom",
     "de": "Ich trinke",
     "deHint": "ئیش ترینکە",
     "en": "I drink",
     "enHint": "ئای درینک"
    },
    {
     "id": "v4",
     "ku": "ئەز دنڤم",
     "kuLat": "Ez dinivim",
     "de": "Ich schlafe",
     "deHint": "ئیش شلافە",
     "en": "I sleep",
     "enHint": "ئای سلیپ"
    },
    {
     "id": "v5",
     "ku": "ئەز فێر دبم",
     "kuLat": "Ez fêr dibim",
     "de": "Ich lerne",
     "deHint": "ئیش لێرنە",
     "en": "I learn",
     "enHint": "ئای لێرن"
    },
    {
     "id": "v6",
     "ku": "ئەز کار دکەم",
     "kuLat": "Ez kar dikem",
     "de": "Ich arbeite",
     "deHint": "ئیش ئاربایتە",
     "en": "I work",
     "enHint": "ئای وۆرک"
    }
   ]
  },
  {
   "id": "unit_10",
   "titleKu": "قۆناغا ١٠: رستەیێن ژیانێ",
   "titleDe": "Einheit 10: Sätze",
   "titleEn": "Unit 10: Real sentences",
   "icon": "chat",
   "color": "#A560E8",
   "concepts": [
    {
     "id": "r1",
     "ku": "ئەز ژ کوردستانێ مە",
     "kuLat": "Ez ji Kurdistanê me",
     "de": "Ich komme aus Kurdistan",
     "deHint": "ئیش کۆمە ئاوس کوردستان",
     "en": "I am from Kurdistan",
     "enHint": "ئای ئێم فرۆم کوردستان"
    },
    {
     "id": "r2",
     "ku": "ئەز ئەلمانی فێر دبم",
     "kuLat": "Ez Almanî fêr dibim",
     "de": "Ich lerne Deutsch",
     "deHint": "ئیش لێرنە دۆیچ",
     "en": "I am learning German",
     "enHint": "ئای ئێم لێرنینگ جێرمەن"
    },
    {
     "id": "r3",
     "ku": "ئەڤرۆ هەوا خۆشە",
     "kuLat": "Evro hewa xweş e",
     "de": "Heute ist das Wetter schön",
     "deHint": "هۆیتە ئیست داس ڤێتەر شۆن",
     "en": "The weather is nice today",
     "enHint": "ذە وێذەر ئیز نایس تودەی"
    },
    {
     "id": "r4",
     "ku": "تو دشێی هاریکاریا من بکەی؟",
     "kuLat": "Tu dişêy harîkariya min bikey?",
     "de": "Kannst du mir helfen?",
     "deHint": "کانست دو میر هێلفن؟",
     "en": "Can you help me?",
     "enHint": "کان یوو هێلپ می؟"
    },
    {
     "id": "r5",
     "ku": "ئەز دچمە بازاڕی",
     "kuLat": "Ez diçime bazarî",
     "de": "Ich gehe zum Markt",
     "deHint": "ئیش گێهە تسوم مارکت",
     "en": "I am going to the market",
     "enHint": "ئای ئێم گۆینگ تو ذە مارکێت"
    },
    {
     "id": "r6",
     "ku": "ئەڤە کتێبا منە",
     "kuLat": "Eve kitêba min e",
     "de": "Das ist mein Buch",
     "deHint": "داس ئیست ماین بوخ",
     "en": "This is my book",
     "enHint": "ذیس ئیز مای بووک"
    }
   ]
  }
 ]
}

## ٣. نموونەیا دەنگێ خۆ بار بکە

ئەڤی دەقی ب دەنگەکێ ئاسایی و ل جهەکێ بێدەنگ بخوینە (د ئەپی دا ل «کلۆنکرنا دەنگی» هەیە و دشێی ژ وێرێ Share بکەی):

> سلاڤ، ئەڤە دەنگێ منە. ئەز ب کوردی دئاخڤم و ئەز دێ ئەلمانی و ئینگلیزی فێر بم. ئەڤرۆ هەوا خۆشە و ئەز زۆر کەیفخۆشم.

ئارەزوومەندانە (بۆ دەربڕینا باشتر): نموونەیەکا ئەلمانی و ئینگلیزی ژی.

In [ ]:
from google.colab import files
print("📤 نموونەیا بادینی (پێدڤی) — Choose files بکە:")
REF_KU = list(files.upload())[0]
REF_DE = REF_EN = None
print("✅", REF_KU)

ئارەزوومەندانە — ئەگەر تە نموونەیا ئەلمانی یان ئینگلیزی نینە، ئەڤان هەردوو خانەیان **دەرباز بکە**.

In [ ]:
print("📤 نموونەیا ئەلمانی:")
REF_DE = list(files.upload())[0]

In [ ]:
print("📤 نموونەیا ئینگلیزی:")
REF_EN = list(files.upload())[0]

## ٤. تاقیکرنەکا خێرا — گوهێ خۆ بدێ

In [ ]:
from IPython.display import Audio, display
from voice_core import *
import tempfile, os

refs = {"ku": Reference(to_wav24k(REF_KU, "ref_ku.wav"), REFERENCE_SCRIPTS["ku"])}
if REF_DE: refs["de"] = Reference(to_wav24k(REF_DE, "ref_de.wav"), REFERENCE_SCRIPTS["de"])
if REF_EN: refs["en"] = Reference(to_wav24k(REF_EN, "ref_en.wav"), REFERENCE_SCRIPTS["en"])

cloner = VoiceCloner()
for lang, text in [("ku", "سپێدە باش، تو چاوانی؟"), ("de", "Guten Morgen, wie geht es dir?"), ("en", "Good morning, how are you?")]:
    wav_to_mp3(cloner.synthesize(text, lang, pick_reference(lang, refs)), f"test_{lang}.mp3")
    print(lang, text); display(Audio(f"test_{lang}.mp3"))

## ٥. دروستکرنا هەمی دەنگان (پاکێتا دەنگی)

In [ ]:
import json, shutil
from pathlib import Path
curriculum = json.loads(Path("curriculum.json").read_text(encoding="utf-8"))
build_pack(curriculum, refs, Path("voice_pack"), cloner)
shutil.make_archive("voice_pack", "zip", "voice_pack")
print("✅ voice_pack.zip ئامادەیە")
files.download("voice_pack.zip")

📱 ئەگەر داگرتن ل مۆبایلێ دەستپێنەکر، ئەڤێ خانێ ▶ بکە. zip دێ چیتە **Google Drive** ـا تە (بوخچا ZimanPro). پاشان ژ ئەپێ Drive داگرە.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
os.makedirs("/content/drive/MyDrive/ZimanPro", exist_ok=True)
shutil.copy("voice_pack.zip", "/content/drive/MyDrive/ZimanPro/voice_pack.zip")
print("✅ Google Drive ➔ ZimanPro ➔ voice_pack.zip")

**پاشان:** ل ئەپێ ZimanPro: `دەنگێ من ➔ 🧬 ➔ هاوردەکرنا voice_pack.zip` ➔ zip هەلبژێرە. 🎉
(ب کۆمپیوتەری: دشێی zip ڤەکەی د `assets/voices/` دا و ژ نوو build بکەی.)

---
## ٦. (ئارەزوومەندانە) سێرڤەرێ زیندی بۆ ئەپی
دا ئەپ ب خۆ ژ مۆبایلێ دەنگێ تە فرێکەت و دەنگێن نوو دروست بکەت. لینکێ `trycloudflare.com` کۆپی بکە و ل ئەپی بنڤیسە.

In [ ]:
!pip -q install fastapi uvicorn python-multipart
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import os, secrets, subprocess, time, re
TOKEN = secrets.token_urlsafe(8)
os.environ["ZIMAN_TOKEN"] = TOKEN
subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"], env=os.environ.copy(),
                 stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: url = m.group(0); break
print("🌍 Server URL :", url)
print("🔑 Token      :", TOKEN)
print("ئەڤان هەردووان ل ئەپی بنڤیسە: دەنگێ من ➔ کلۆنکرنا دەنگی")